# 10 — Video-Only Baseline

Define a compact CNN-based video baseline to establish performance using visual evidence alone.

> **Project:** Multimodal Deepfake Detection using LAV-DF (audio + video). Run cells in order. Confirm dataset paths and split definitions before training. Never fit preprocessing or tune thresholds on the test split.

## Experimental design
Input: a fixed number of resized RGB frames per clip. A lightweight 2D CNN encodes each frame; average pooling across frames produces a clip representation. This baseline provides a visual-only reference for later fusion.

In [ ]:
import torch
import torch.nn as nn

class SmallFrameCNN(nn.Module):
    def __init__(self, feature_dim=128):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(64, feature_dim), nn.ReLU()
        )
    def forward(self, frames):
        # frames: [batch, time, channels, height, width]
        b, t, c, h, w = frames.shape
        encoded = self.features(frames.reshape(b*t, c, h, w))
        return encoded.reshape(b, t, -1).mean(dim=1)

class VideoOnlyClassifier(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        self.encoder = SmallFrameCNN()
        self.head = nn.Linear(128, num_classes)
    def forward(self, frames):
        return self.head(self.encoder(frames))

model = VideoOnlyClassifier()
print("Parameters:", sum(p.numel() for p in model.parameters()))

## Before training
Implement a dataset class that reads cached frame arrays and labels by stable ID. Use a small batch size, mixed precision where supported, gradient clipping if needed, checkpoints, and a CSV log. Select hyperparameters using the development split only.